In [6]:
import numpy as np

In [4]:
def loss(x, w, y):
    l = 1/(2*len(x)) * ((w * x - y) ** 2)
    return l

In [3]:
def grad(x, w, y):
    g = 1/N * x.T * (x * w - y)
    return g

In [8]:
def adagrad(w, g, G, alpha=0.01, epsilon=1e-8):
    G = G + g ** 2
    w = w - alpha / (n.sqrt(G) + epsilon) * g

In [16]:
import numpy as np
import urllib.request

def load_mnist_direct():
    url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
    filename = "mnist.npz"
    
    # Download if not present
    import os
    if not os.path.exists(filename):
        print("Downloading MNIST...")
        urllib.request.urlretrieve(url, filename)
    
    # Load using pure NumPy
    with np.load(filename) as data:
        train_X, train_y = data['x_train'], data['y_train']
        test_X, test_y = data['x_test'], data['y_test']
        
    return train_X, train_y, test_X, test_y

# Load data
train_images, train_labels, test_images, test_labels = load_mnist_direct()
print(f"Loaded {len(train_images)} images. Shape: {train_images[0].shape}")

Loaded 60000 images. Shape: (28, 28)


In [19]:
def get_patches(image):
    """Turns 28x28 image into (26*26, 9) patches for 3x3 filters"""
    h, w = image.shape
    patches = []
    for i in range(h - 2):
        for j in range(w - 2):
            patches.append(image[i:i+3, j:j+3].flatten())
    return np.array(patches)

# --- INITIALIZATION ---

# 1. Conv weights (8 filters of size 3x3)
W1 = np.random.randn(9, 8) * 0.1
G1 = np.zeros_like(W1)

# 2. Dense weights (Takes flattened 8 filtered 26x26 images to 10 classes)
W2 = np.random.randn(26 * 26 * 8, 10) * 0.1
G2 = np.zeros_like(W2)

# --- TRAINING STEP ---

def train_one_image(img, label_idx, lr=0.001):
    global W1, G1, W2, G2
    
    # Pre-process image (Normalizing 0-255 to 0-1)
    x_input = img / 255.0
    
    # 1. Forward Pass
    patches = get_patches(x_input)
    # Convolution via YOUR matrix logic
    z1 = np.dot(patches, W1) 
    a1 = np.maximum(0, z1) # ReLU activation
    
    # Flatten for Dense layer
    x_dense = a1.flatten().reshape(1, -1)
    
    # Prepare one-hot target for your grad function
    y_true = np.zeros((1, 10))
    y_true[0, label_idx] = 1.0
    
    # 2. Backward Pass using YOUR functions
    # Update the Classifier (Dense) layer
    g_W2 = grad(x_dense, W2, y_true)
    W2, G2 = adagrad(W2, g_W2, G2, lr)
    
    # Backpropagate error to the Conv layer
    # Treat the layer below as a linear problem for your 'grad' function
    error_signal = np.dot((np.dot(x_dense, W2) - y_true), W2.T).reshape(z1.shape)
    g_W1 = grad(patches, W1, z1 - error_signal)
    W1, G1 = adagrad(W1, g_W1, G1, lr)
    
    return loss(x_dense, W2, y_true)

In [22]:
from tqdm.notebook import tqdm
import numpy as np

# Settings
epochs = 5
learning_rate = 0.001

print(f"Starting training for {epochs} epochs...")

for epoch in range(epochs):
    # 1. Shuffle data at the start of each epoch (Crucial for research)
    indices = np.arange(len(train_images))
    np.random.shuffle(indices)
    
    # 2. Setup tqdm progress bar for this epoch
    # We use 'total' to show the number of images, and 'desc' for the epoch count
    pbar = tqdm(total=len(train_images), desc=f"Epoch {epoch + 1}/{epochs}")
    
    epoch_loss = 0
    correct = 0

    for i, idx in enumerate(indices):
        img = train_images[idx]
        label = train_labels[idx]
        
        # --- Run your training logic ---
        # Note: I'm using your 'train_one_image' function logic here
        l = train_one_image(img, label, lr=learning_rate)
        
        # --- Update Metrics ---
        epoch_loss += l
        
        # Quick forward pass to check accuracy for the progress bar
        # (This uses your matrix-based prediction logic)
        x_norm = img / 255.0
        patches = get_patches(x_norm)
        out = np.dot(np.maximum(0, np.dot(patches, W1)).flatten(), W2)
        if np.argmax(out) == label:
            correct += 1
            
        # 3. Update the progress bar every 10 images (to save processing time)
        if i % 10 == 0:
            current_acc = (correct / (i + 1)) * 100
            # .set_postfix shows metrics on the right side of the bar
            pbar.set_postfix(loss=f"{l:.4f}", acc=f"{current_acc:.1f}%")
            pbar.update(10) # Advance the bar by 10 steps

    pbar.close() # Finish the bar for this epoch

Starting training for 5 epochs...


Epoch 1/5:   0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 2/5:   0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 3/5:   0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 4/5:   0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 5/5:   0%|          | 0/60000 [00:00<?, ?it/s]